# Faithfulness is architecture-independent

**Claim:** ECG+PPG blood-pressure models can be accurate yet *not faithful* to the physiology they should use (pulse transit time, PTT). Here we show it for **three architectures** — an MLP, a 1D-CNN, and a small Transformer — with a causal audit.

**Method.** Train each model to predict [SBP, DBP] from ECG+PPG. Then run the **causal PTT-shift audit**: roll the PPG channel by ±Δ samples (a later PPG = a longer transit time) and measure how predicted BP responds. Physiologically faithful = BP goes **down** when PTT goes **up** (`dBP/dPTT < 0`; `frac_phys > 0.5`).

*Reproduce:* run top-to-bottom (a few minutes on CPU). Data is the bundled `data/vitaldb_mini.npz`; the audit lives in `mechlib.py`.

In [ ]:
import numpy as np, torch, torch.nn as nn
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # repo root, for mechlib + data/
import mechlib

torch.manual_seed(0); np.random.seed(0)
dev = torch.device("cpu"); ECG, PPG = 0, 1
d = mechlib.load_mini("../data/vitaldb_mini.npz"); fs = int(d["fs"])
Xtr = mechlib.normalize(d["Xtr"][:, :, [ECG, PPG]]); ytr = d["ytr"]
Xte = mechlib.normalize(d["Xte"][:, :, [ECG, PPG]]); yte = d["yte"]; gte = d["gte"]
L = Xtr.shape[1]
print("train", Xtr.shape, " test", Xte.shape, " fs", fs)

In [ ]:
class MLP(nn.Module):
    def __init__(s, D=64):
        super().__init__()
        s.body = nn.Sequential(nn.AvgPool1d(10), nn.Flatten(),          # 1250 -> 125 / channel
                               nn.Linear(2*(L//10), 128), nn.ReLU(), nn.Linear(128, D), nn.ReLU())
        s.head = nn.Linear(D, 2)
    def forward(s, x): return s.head(s.body(x.transpose(1, 2)))


class CNN(nn.Module):
    def __init__(s, w=32):
        super().__init__()
        s.enc = nn.Sequential(nn.Conv1d(2, w, 7, 2, 3), nn.ReLU(), nn.Conv1d(w, w*2, 7, 2, 3),
                              nn.ReLU(), nn.Conv1d(w*2, w*2, 7, 2, 3), nn.ReLU(),
                              nn.AdaptiveAvgPool1d(1), nn.Flatten())
        s.head = nn.Linear(w*2, 2)
    def forward(s, x): return s.head(s.enc(x.transpose(1, 2)))


class Transformer(nn.Module):
    def __init__(s, dm=64, patch=25, heads=4, depth=2):
        super().__init__()
        s.patch = nn.Conv1d(2, dm, patch, patch)                       # -> L/patch tokens
        s.pos = nn.Parameter(torch.randn(1, L//patch, dm) * 0.02)
        s.tr = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(dm, heads, dm*2, batch_first=True), depth)
        s.head = nn.Linear(dm, 2)
    def forward(s, x):
        t = s.patch(x.transpose(1, 2)).transpose(1, 2) + s.pos
        return s.head(s.tr(t).mean(1))


def train(model, epochs=20, bs=128):
    opt = torch.optim.Adam(model.parameters(), 2e-3)
    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr); var = torch.tensor(ytr.var(0))
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            b = perm[i:i+bs]; loss = (((model(Xt[b])-yt[b])**2)/var).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); return model

In [ ]:
# accuracy (calibrated DBP MAE) + faithfulness (causal PTT-shift audit) per architecture
print(f"{'model':>12} | {'DBP MAE':>8} {'baseline':>9} | {'dBP/dPTT':>9} {'frac phys':>10} | verdict")
print("-" * 74)
base = mechlib.calibrated_mae(np.repeat(ytr.mean(0)[None], len(yte), 0), yte, gte, K=3)[1]
for name, M in [("MLP", MLP()), ("1D-CNN", CNN()), ("Transformer", Transformer())]:
    train(M)
    pred = mechlib.predict(M, Xte, dev)
    mae = mechlib.calibrated_mae(pred, yte, gte, K=3)[1]
    au = mechlib.causal_ptt_audit(M, Xte, fs, dev, ppg_pos=1)           # roll PPG -> dBP/dPTT
    slope, frac = au["dbp"]["dBP_dPTT"], au["dbp"]["frac_correct_sign"]
    verdict = "FAITHFUL" if (slope < 0 and frac > 0.5) else "not faithful"
    print(f"{name:>12} | {mae:7.1f}  {base:8.1f} | {slope:+9.2f} {frac:10.2f} | {verdict}")

## What this teaches us about *building* models

1. **Accuracy is not faithfulness.** All three architectures are trained BP predictors, yet every one has the **wrong-signed** PTT response. Model selection needs a causal audit, not just MAE.
2. **When the true mechanism is weak in the data, models grab the nearest confound** — here the heart-rate / cardiac-timing shortcut (see `app_faithfulness.py` tab 3, the mechanism battery). Swapping architectures does not remove the shortcut.
3. **So "build a better model" really means "fix the data + objective":** use interventional / BP-manipulation data where PTT actually drives BP, or constrain the model (penalize dependence on HR, or force prediction through a physiological bottleneck such as the reconstructed pressure wave).
4. The **synthetic sandbox** (tab 1) validates that the audit *can* read faithful when the mechanism is genuinely used — so these nulls are real verdicts, not a blind instrument.